# Ensemble Learning and Random Forests

When a complex question is asked to a large number of random people, and the answer is aggregated, the results are usualy better than an expert's answer. This is called _wisdom of the crowd_. Similarly, if we aggregate the predictions of a group of predictors, we will make better predictions than a the best individual predictors. A group of predictors is alled an _ensemble_, and this technique is known as Ensemble Learning, and an Ensemble Learning Algorithm is called an Ensemble method. 

An ensemble of Decision Trees is known as _Random Forest_. These are one of the most powerful ML algorithms available today, despite it's simplicity.

## Voting Classifiers

Suppose we have multiple types of classifiers with ~80% accuracy in each. A simple way to create a better classifier is to aggregate the predictions of each classifier and predict the class that gets the most votes. This is called **hard voting classifier**. 

This voting classifier often achieves a higher accuracy that the best classifier in the ensemble. In fact, even if each classifier is a weak learner (only slightly better than guessing), the ensemble can still be a strong learner, provided there are a sufficient number of weak learners and they are sufficiently diverse. 

This can be explained with something called as _law of large numbers_. If a biased coin (51% heads, 49% tails) is tossed a large number of times, the probablity of obtaining majority of heads keeps on increasing. So if an ensemble has 1000 classifiers which have a 51% accuracy (or any number just better than random), the voting classifier can have upto 75% accuracy.

**Note**: Ensemble methods work best when predictors are as independent from one another as possible. 

In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [4]:
from sklearn.metrics import accuracy_score

for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.85
RandomForestClassifier 0.9
SVC 1.0
VotingClassifier 0.9


### Soft Voting

If all classifiers are able to estimate class probablities (with `predict_proba()` method), then Scikit-Learn can predict the class with the highest class probablity, averaged out over all individual classifiers. This is called _soft voting_. 

It often achieves higher performance than hard voting because it gives more weight to highly confident votes. 

## Bagging and Pasting

- When sampling is performed with replacement, this method is called _bagging_ (short for Bootstrap AGGregating).
- When sampling is performed without replacement, it called _pasting_.
- Bagging and pasting involves training several predictors on different random samples of the training set.
- The ensemble can make a prediction for a new instance by simply aggregating the predictions of all predictors. The aggregation is typically the statistical mode for classification, and the average for regression.
- Aggregation reduces both bias and variance. 

In [7]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=min(100, X_train.shape[0]), bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

In [8]:
accuracy_score(y_test, y_pred)

0.85

## Out-of-Bag Evaluation

- During bagging, some instances may be sampled several times for any given predictor, while others may not be sampled at all.
- By default a BaggingClassifier samples `m` training instances with replacement (`bootstrap=True`), where m is the size of the training set. This means that only about 63% of the training instances are sampled on average for each predictor.6 The remaining 37% of the training instances that are not sampled are called out-of-bag (oob) instances.
- Since a predictor never sees the oob instances during training, it can be evaluated on these instances, without need for a separate validation set. 

In [9]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500, 
    bootstrap=True, n_jobs=-1, oob_score=True)

bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=500,
                  n_jobs=-1, oob_score=True)

In [10]:
bag_clf.oob_score_

0.9625

In [11]:
from sklearn.metrics import accuracy_score

y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.85

In [12]:
bag_clf.oob_decision_function_

array([[0.        , 1.        ],
       [0.99408284, 0.00591716],
       [1.        , 0.        ],
       [0.99462366, 0.00537634],
       [0.56593407, 0.43406593],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [1.        , 0.        ],
       [0.9086758 , 0.0913242 ],
       [0.        , 1.        ],
       [0.99489796, 0.00510204],
       [0.        , 1.        ],
       [1.        , 0.        ],
       [0.13978495, 0.86021505],
       [0.00588235, 0.99411765],
       [0.04081633, 0.95918367],
       [0.96774194, 0.03225806],
       [0.09090909, 0.90909091],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [1.        , 0.        ],
       [0.87027027, 0.12972973],
       [0.98477157, 0.01522843],
       [0.12182741, 0.87817259],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.92178771, 0.07821229],
       [0.6       , 0.4       ],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.

## Random Patches and Random Subspaces

- `BaggingClassifier` supports sampling the features as well. Sampling is controlled by two hyperparameters: `max_features` and `bootstrap_features`. They work the same way as `max_samples` and `bootstrap` but for feature sampling instead of instance sampling. Thus, each predictor will be trained on a random subset of the input features.
- This technique is useful with high-dimensional inputs (like images).
- Sampling both training instances and features is called the _Random Patches method_.
- Keeping all training instances (`bootstrap=False`; `max_samples=1.0`) but sampling features (`bootstrap_features=True`; `max_features<1.0` is called _Random Subspaces method_.  

## Random Forests